# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Make sure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print top-level metadata information
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Number of record sets: {len(metadata.record_sets)}")

## 2. Data Overview
Review available record sets, fields, and their IDs. For each record set, we will display its `@id`, human-readable name, and list field/column `@id`s.

In [ ]:
# Utility function to recursively display record sets and their fields
def display_record_set_info(ds):
    for rs in ds.metadata.record_sets:
        print(f"RecordSet @id: {rs.id}")
        print(f"  Name: {getattr(rs, 'name', 'N/A')}")
        # Fields
        if hasattr(rs, 'fields') and rs.fields:
            print("  Field @ids:")
            for f in rs.fields:
                print(f"    - {f.id}")
        # Columns (for Table-oriented record sets)
        if hasattr(rs, 'columns') and rs.columns:
            print("  Columns @ids:")
            for c in rs.columns:
                print(f"    - {c.id}")
        print("\n-----------------------\n")

display_record_set_info(dataset)

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis. Here we will extract the first RecordSet by its `@id` for demonstration.

**Note:** We'll use the `@id` fields for referencing as per Croissant specification.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Record sets in dataset:")
for i, rs_id in enumerate(record_set_ids):
    print(f"  {i+1}. {rs_id}")

# Extract all record sets as DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet {record_set_id}")

# Example: Inspect the columns of the first record set
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print("\nSample columns for RecordSet @id:", first_record_set_id)
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Explore the loaded data. As an example, we'll select a numeric field (by `@id`) and perform:
 - Filtering for records above a threshold
 - Normalizing that field
 - Grouping by a categorical field, if present

Please see the printed columns for selecting appropriate `@id` values. Update variable values as needed for dataset specifics.

In [ ]:
# Choose one record set (using @id)
record_set_id = first_record_set_id  # Or replace with another specific record set @id
df = dataframes[record_set_id]
print(f"Columns available in {record_set_id}:")
print(df.columns.tolist())

# Choose a numeric field (@id) for demonstration
# Example: The dataset likely has an 'Age' field. Let's list candidate numeric columns.
import re
numeric_fields = [col for col in df.columns 
                 if re.search(r"age|count|number|interval|years", col, re.IGNORECASE) or pd.api.types.is_numeric_dtype(df[col])]
print("Possible numeric fields:", numeric_fields)

# For demonstration, use the first detected numeric field
numeric_field_id = numeric_fields[0] if numeric_fields else df.columns[0]  # fallback
# Filtering: Only include records where numeric_field > threshold
threshold = 60  # Update this threshold based on exploration
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold].copy()
else:
    # Try to convert to numeric (if stored as str)
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
print(f"Filtered records for {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field in the filtered DataFrame
filtered_df[numeric_field_id + "_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - 
    pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

# Now group by a categorical field (by @id), e.g. 'Sex', 'Location', etc.
possible_group_fields = [col for col in df.columns 
                        if any(word in col.lower() for word in ["sex", "gender", "location", "site", "stage", "msi"])]
print("Possible group fields:", possible_group_fields)
if possible_group_fields:
    group_field_id = possible_group_fields[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name="mean_" + numeric_field_id)
    print(f"Grouped by {group_field_id} (mean {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships using matplotlib or seaborn.

Let's plot the normalized numeric field's distribution, and if grouping variable is present, compare means across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the normalized numeric field
if not filtered_df.empty:
    fig, ax = plt.subplots(1,1, figsize=(7,4))
    sns.histplot(filtered_df[numeric_field_id + "_normalized"].dropna(), bins=15, kde=True, ax=ax)
    ax.set_title(f"Distribution of Normalized {numeric_field_id}")
    plt.xlabel(numeric_field_id + " (normalized)")
    plt.ylabel("Frequency")
    plt.show()
    
    # If a group field was found, plot group means
    if 'group_field_id' in locals():
        group_means = (
            filtered_df.groupby(group_field_id)[numeric_field_id]
            .mean().sort_values(ascending=False)
        )
        plt.figure(figsize=(8,4))
        group_means.plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
We have demonstrated loading and inspecting the FAIR² clinical dataset via the Croissant schema, explored its record sets and fields via their `@id`s, and performed basic processing and visualization.

- Use field and record set `@id`s for precise and reproducible data referencing as per the Croissant standard.
- See `dataframes` for full access to all data for further custom analyses.
- Consult the schema and printed column names to select the most relevant fields for your research or processing tasks.

For more advanced analyses (survival analysis, statistical modeling, etc.), follow up with domain-specific libraries and methods.

Happy analyzing!